# Quick start: simulating single-cell counts with `NegBinCopula`

This notebook introduces the main `scDesigner` workflow: specify a count model, fit it to an `AnnData` object, inspect the fitted parameters, and simulate under either the observed or a new covariate design.

The example comes from the [scVelo pancreas dataset](https://scvelo.readthedocs.io/en/stable/scvelo.datasets.pancreas.html), which captures pancreatic endocrinogenesis. This subset contains 2,087 cells from a single trajectory and 1000 highly variable genes.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from scdesigner.datasets import pancreas

example_sce = pancreas() # the current pancreas dataset only has 100 genes
example_sce

AnnData object with n_obs × n_vars = 2087 × 100
    obs: 'clusters_coarse', 'clusters', 'S_score', 'G2M_score', 'cell_type', 'sizeFactor', 'pseudotime'
    var: 'highly_variable_genes'
    uns: 'X_name', 'clusters_coarse_colors', 'clusters_colors', 'day_colors', 'neighbors', 'pca'
    obsm: 'PCA', 'UMAP', 'X_pca', 'X_umap'
    layers: 'counts', 'cpm', 'logcounts', 'spliced', 'unspliced'
    obsp: 'connectivities', 'distances'

## 1. Specify simulator

The simulators take in cell-by-gene `AnnData` objects for model fitting, where they use counts information in `X` as the target variable and expect the covariates referenced by a formula to be columns from `obs`. 

`NegBinCopula` accepts `AnnData` object with **raw, nonnegative integer counts in `X`**, and it has three formulas, all of which default to `"~ 1"`:

- `mean_formula` models the expected count for every cell and gene.
- `dispersion_formula` models the negative-binomial size parameter. Under this parameterization, the variance is $\mu + \mu^2/r$, so smaller $r$ means more variation beyond a Poisson model.
- `copula_formula` determines whether one or several gene-correlation matrices are estimated. It currently supports categorical grouping variables; `"~ 1"` estimates one shared correlation structure.

Formulas support transformations such as `bs()` for B-splines. Here, a spline allows gene means to change smoothly and nonlinearly over developmental pseudotime. We keep dispersion and correlation constant to avoid adding complexity.

In [2]:
from scdesigner.simulators import NegBinCopula

sim = NegBinCopula(
    mean_formula="~ bs(pseudotime, df=5)",
    dispersion_formula="~ 1",
    copula_formula="~ 1",
)

## 2. Fit the marginal and copula models

Similar to `scDesign3`, `scDesigner` also has two steps (parallel marginal fitting and copula fitting) in the model training process, whereas the `fit()` function carry them out in a consecutive manner by first optimizing the marginal models and then estimating the covariance. 

Training stops when that loss fails to improve by `loss_tol` for `patience` epochs after `min_epochs`; the best marginal checkpoint is restored before the copula is fitted. See the [early-stopping example](early_stopping_example.ipynb) for more detail.

In [3]:
# Seed both libraries because optimization and copula fitting use random numbers.
np.random.seed(0)
torch.manual_seed(0)

sim.fit(
    example_sce,
    max_epochs=500,
    val_frac=0.1,
    min_epochs=20,
    patience=10,
    loss_tol=1e-4,
    validation_seed=0,
    batch_size=1024,
    lr=1e-3,
    verbose=False,
)

Estimating copula correlation: 100%|██████████| 3/3 [00:00<00:00,  9.74it/s]


The fit history records the marginal training and validation losses. It is worth checking that the best epoch was not the first epoch, the losses are finite, and validation loss did not deteriorate strongly while training loss continued to fall.

In [4]:
print(
    f"best epoch: {sim.marginal.best_epoch}; "
    f"stopped epoch: {sim.marginal.stopped_epoch}"
)
sim.marginal.fit_history_df.tail()

best epoch: 257; stopped epoch: 267


,epoch,train_loss,val_loss,best,stopped
262,263,1.870078,1.876753,False,False
263,264,1.870060,1.876734,False,False
264,265,1.870043,1.876716,False,False
265,266,1.870026,1.876698,False,False
266,267,1.870009,1.876680,False,True


For thousands of genes, full copula estimation can be expensive and may produce a non-positive-definite correlation matrix, especially within small sample size. Pass `top_k=<number>` to `fit()` to model a full correlation block only for the most highly expressed genes; the remaining genes retain fitted negative-binomial marginals but are treated as independent in the copula. 

See the [large dataset example](vignettes-large-scale.ipynb) for more detail on `scDesigner`'s scalability design.

## 3. Inspect fitted values and parameters

`predict()` returns the fitted marginal parameters for each cell-gene pair in dictionary format. It uses the training covariates by default when no `obs` values are specified; we can also specify new covariate values to predict the means and dispersions. 

The fitted coefficients and copula structures are available through `sim.parameters`, `sim.marginal`, and `sim.copula` (see [distributions overview](distributions_overview.ipynb))

In [5]:
fitted = sim.predict()
fitted['mean']

array([[4.8407379e+01, 1.8448949e+00, 5.5794674e+01, ..., 4.3633088e-01,
        1.3374237e-03, 1.2136539e+00],
       [1.2564061e+00, 1.5454294e-01, 5.7681947e+00, ..., 1.9728613e+00,
        9.0026306e-03, 3.8604097e+00],
       [1.1042807e+02, 8.7116327e+00, 4.4998951e+01, ..., 4.4377929e-01,
        3.3875615e-03, 6.7330551e-01],
       ...,
       [6.4041885e+01, 2.8666463e+00, 5.3730453e+01, ..., 4.2683348e-01,
        1.5948187e-03, 1.0133742e+00],
       [1.3218869e+01, 4.6081355e-01, 4.7346523e+01, ..., 6.1435080e-01,
        1.5211997e-03, 2.3320646e+00],
       [5.8476835e-01, 1.9426128e-01, 2.5213832e-01, ..., 3.7348113e+00,
        1.3912406e-02, 2.2023039e+00]], dtype=float32)

In [6]:
# Rows are formula terms and columns are genes.
sim.parameters["marginal"]["mean"].iloc[:, :5]

,Pyy,Iapp,Chgb,Rbp4,Spp1
Intercept,-0.628266,-1.775196,1.017537,-0.459690,3.880537
"bs(pseudotime, df=5)[1]",1.352797,2.312887,-12.853308,0.267358,1.696402
"bs(pseudotime, df=5)[2]",-1.908343,-1.118718,1.017034,-3.123623,-5.136424
"bs(pseudotime, df=5)[3]",5.332261,0.848964,4.120718,4.839564,-6.980855
"bs(pseudotime, df=5)[4]",6.538476,7.932713,1.712499,4.609818,-4.260563
"bs(pseudotime, df=5)[5]",5.089781,7.862651,1.491324,3.887867,-5.182944


`complexity()` reports marginal and copula AIC/BIC. These criteria can help compare candidate formula specifications, but comparisons are meaningful only when models are fitted to the same cells and genes.

In [7]:
sim.complexity(batch_size=1024)

Computing log-likelihood...: 100%|██████████| 3/3 [00:00<00:00, 18.90it/s]


{'marginal_aic': 782288.5625,
 'marginal_bic': 786239.000534954,
 'copula_aic': -9285.13354155235,
 'copula_bic': 18650.106848479794}

## 4. New count matrix simulation

Similar to `predict()`, Calling `sample()` without `obs` generates a count matrix based on training `obs` data, and calling `sample()` with new `obs` values gives a count matrix with the new corresponding mean and dispersion parameters. 

The output is an `AnnData` object where `.X` contains the simulated counts, and `.obs` and `.var` carry the template metadata needed to interpret them.

In [8]:
np.random.seed(1)
sampled_sce = sim.sample()
sampled_sce

AnnData object with n_obs × n_vars = 2087 × 100
    obs: 'clusters_coarse', 'clusters', 'S_score', 'G2M_score', 'cell_type', 'sizeFactor', 'pseudotime'
    var: 'highly_variable_genes'

## 5. Simulation validation

A successful fit should reproduce the features targeted by the model—not necessarily every property of the original dataset. At minimum, compare gene means, variances, zero fractions, library sizes, and correlations. Here we only compared the UMAP as a visual check.

In [9]:
from scdiagnostics import compare_umap

compare_umap(example_sce, sampled_sce, color="pseudotime")

/Users/pyl/anaconda3/envs/scdesigner/lib/python3.11/site-packages/anndata/_core/storage.py:39: ImplicitModificationWarning: X should not be a np.matrix, use np.ndarray instead.
  warnings.warn(msg, ImplicitModificationWarning)
/Users/pyl/anaconda3/envs/scdesigner/lib/python3.11/site-packages/scdiagnostics/data.py:39: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  return real_.concatenate(simulated_, join="outer", batch_key=None)


alt.FacetChart(...)